# 🦎 Lizard Species Classification — Thomas More AI Challenge

**Team members:** *(vul hier jullie namen in)*

**Course:** Artificial Intelligence — Thomas More

**Challenge:** Classify lizard images into **7 species**. Kaggle scoring metric: **F1 (macro)**.

---

## Introduction

This notebook builds an image classifier that can distinguish seven similar-looking lizard species: *Black spiny-tailed iguana*, *Brown anole*, *Cuban knight anole*, *Desert iguana*, *Green anole*, *Green iguana*, and *Lesser Antillean iguana*. With only ~1300 training images this is a **small-data** problem, which directly shapes our strategy: we cannot train a deep CNN from scratch, so we lean heavily on **transfer learning** and **data augmentation**.

The Kaggle metric is **F1 (macro)** — the unweighted mean of per-class F1 scores. This is stricter than accuracy: getting one class systematically wrong costs us a lot, even if the other six are perfect. Our evaluation reports both accuracy and F1-macro so we can spot this kind of failure early.

Our overall plan:

1. **EDA** — explore the dataset (class balance, image samples, sizes).
2. **Data preparation** with `image_dataset_from_directory` and an 85/15 train/val split.
3. **In-model data augmentation** (flip, rotation, zoom, translation, contrast) to artificially enlarge the training set and reduce overfitting.
4. **Transfer learning with EfficientNetV2S** — a strong ImageNet-pretrained backbone.
5. **Two-phase training:** frozen feature extraction first, then fine-tuning the top layers of the base.
6. **Cosine learning-rate schedule** for fine-tuning (listed as a bonus enhancement in the brief).
7. **Optimal epoch analysis** plus a confusion matrix and a per-class F1 breakdown.
8. **Test-Time Augmentation (TTA)** when generating the Kaggle submission.
9. **GenAI section** — required by the brief.

---
## 1. Setup & imports

Before doing anything else we import all the libraries we will need throughout the notebook, fix the random seed for reproducibility, and define the paths and hyper-parameters in one central place. Putting hyper-parameters at the top of the notebook (rather than scattering them across cells) makes it trivial to re-run an experiment with different settings later.

### 1.1 Imports & reproducibility

We import TensorFlow / Keras for the model, `image_dataset_from_directory` for loading the data, EfficientNetV2S as the pre-trained backbone, scikit-learn for the evaluation metrics (confusion matrix, classification report, F1 score), and the usual numpy / pandas / matplotlib stack for everything else.

Setting a single fixed seed (`42`) for `random`, `numpy`, and TensorFlow makes the notebook *reproducible*: the same code on the same machine produces the same numbers. Without this, comparing two experiments ("did my change actually help, or did I just get lucky with a different shuffle?") becomes guesswork.

In [ ]:
import os, math, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, optimizers
from tensorflow.keras.utils import image_dataset_from_directory
from tensorflow.keras.applications import EfficientNetV2S
from tensorflow.keras.applications.efficientnet_v2 import preprocess_input
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, LearningRateScheduler

from sklearn.metrics import (confusion_matrix, ConfusionMatrixDisplay,
                             classification_report, f1_score)

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
keras.utils.set_random_seed(SEED)

print('TensorFlow:', tf.__version__)
print('GPU(s)    :', [g.name for g in tf.config.list_physical_devices('GPU')] or 'CPU only')

### 1.2 Paths & hyper-parameters

We put every path and hyper-parameter in one cell so the rest of the notebook never has hard-coded magic numbers. The choices below are deliberate:

- `IMG_SIZE = 224` — the native input size EfficientNetV2S was pretrained on. Using exactly the pretraining resolution avoids any quality loss from up- or down-sampling and lets the model use its features as-is.
- `BATCH_SIZE = 32` — a standard mid-range choice. Larger batches give smoother gradients but each step is slower; smaller batches are noisier but make more updates per epoch. 32 is a good compromise for a dataset this small.
- `EPOCHS_P1 = 12` — phase 1 only trains a tiny new head, so it converges quickly. Twelve is a generous upper bound; `EarlyStopping` will cut it short.
- `EPOCHS_P2 = 25` — phase 2 (fine-tuning) needs more epochs because we're now updating millions of parameters with a tiny learning rate.
- `LR_P1 = 1e-3` — the standard Adam learning rate for training a fresh head from scratch.
- `LR_P2_MAX = 1e-4` and `LR_P2_MIN = 1e-6` — the peak and floor of the cosine schedule used during fine-tuning. The big drop (two orders of magnitude) lets the model leave a bad local minimum at the start, then settle precisely into a better one.
- `FINE_TUNE_AT = 100` — keep the first 100 layers of EfficientNetV2S (low-level features like edges and textures) frozen during fine-tuning. Only the higher-level layers get adapted to lizard imagery.

We also derive `CLASS_NAMES` directly from the folder structure. Because we sort alphabetically, the resulting order matches the integer labels in `train.csv` exactly — so label `3` always means the same species everywhere in the notebook.

In [ ]:
BASE_PATH  = 'lizard-prediction-thomas-more'
TRAIN_DIR  = os.path.join(BASE_PATH, 'train')
TEST_DIR   = os.path.join(BASE_PATH, 'test')
TRAIN_CSV  = os.path.join(BASE_PATH, 'train.csv')
TEST_CSV   = os.path.join(BASE_PATH, 'test.csv')

CLASS_NAMES = sorted([c for c in os.listdir(TRAIN_DIR) if not c.startswith('.')])
NUM_CLASSES = len(CLASS_NAMES)
print('Class names :', CLASS_NAMES)

IMG_SIZE     = 224
BATCH_SIZE   = 32
EPOCHS_P1    = 12
EPOCHS_P2    = 25
LR_P1        = 1e-3
LR_P2_MAX    = 1e-4
LR_P2_MIN    = 1e-6
FINE_TUNE_AT = 100

OUTPUT_DIR = 'project_outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)

---
## 2. Exploratory Data Analysis (EDA)

Before building any model, it is essential to understand the dataset. The questions we want to answer are:

1. **How much data do we have, and how is it labelled?** This affects whether we need transfer learning at all, and which loss function to use.
2. **Is the dataset balanced across classes?** An imbalanced dataset would force us to use `class_weight` or oversampling, otherwise a naive model just learns to predict the majority class.
3. **What do the images look like?** Some pairs of species may be visually almost identical, which sets our expectations for which classes the model will struggle with.
4. **What are the image sizes?** This determines the resize strategy and tells us whether we are losing or inventing information when we feed the model 224×224 inputs.

Skipping EDA and jumping straight to modelling is one of the most common ways to waste time on a project — you can spend hours debugging a model that is actually performing correctly on broken data.

### 2.1 Load the CSVs

`train.csv` contains the (filename → label) mapping that we use to verify the directory structure and sanity-check the label distribution. `test.csv` contains the IDs of the images Kaggle wants us to predict; we will need this list when generating the submission file. We just look at the shapes and a few rows here — no data is actually loaded into the model yet.

In [ ]:
train_df = pd.read_csv(TRAIN_CSV)
test_df  = pd.read_csv(TEST_CSV)

print(f'Train: {train_df.shape[0]} images')
print(f'Test : {test_df.shape[0]} images')
print(f'Unique labels: {sorted(train_df["label"].unique())}')
train_df.head()

### 2.2 Class distribution

We count how many images each species has and show the result as a bar chart. The colour and labels are styled for readability. If a single class had, say, 3× more images than the rest, the model would be biased toward predicting it (because just guessing that majority class would give a high accuracy on the training set). The fix in that case would be `class_weight={ ... }` in `model.fit()`, or oversampling the rare classes. This cell tells us whether we need to do that.

In [ ]:
unique_labels, counts = np.unique(train_df['label'], return_counts=True)
print('Counts per class:', dict(zip(unique_labels, counts)))
print(f'Min/max per class: {counts.min()}/{counts.max()}')

fig, ax = plt.subplots(figsize=(11, 4))
ax.bar(unique_labels, counts, color='steelblue')
ax.set_xticks(unique_labels)
ax.set_xticklabels([CLASS_NAMES[i].replace('_', ' ') for i in unique_labels],
                   rotation=20, ha='right')
ax.set_ylabel('Number of training images')
ax.set_title('Class distribution (training set)')
for x, c in zip(unique_labels, counts):
    ax.text(x, c + 1, str(c), ha='center', fontsize=9)
plt.tight_layout(); plt.show()

**Observation:** Counts range from 178 to 199 per class — a spread of less than 12%. The dataset is well-balanced, so we can use plain accuracy and F1-macro without any rebalancing. We do *not* pass `class_weight` to `model.fit()`.

### 2.3 Sample image per class

Showing one example image from each species serves two purposes. First, it is a sanity check that the folder structure is what we think it is — that the *Brown anole* folder really contains brown anoles and not, say, mislabelled iguanas. Second, it gives us a feel for *task difficulty*. Some pairs of species (Brown vs Green anole, Cuban knight anole vs Green anole) look similar even to a human; others (Desert iguana vs Brown anole) are obviously different. We expect the confusion matrix later to reflect exactly these visual similarities.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(15, 7))
axes = axes.flatten()
for i, cname in enumerate(CLASS_NAMES):
    cdir = os.path.join(TRAIN_DIR, cname)
    sample_file = os.listdir(cdir)[0]
    img = mpimg.imread(os.path.join(cdir, sample_file))
    axes[i].imshow(img)
    axes[i].set_title(cname.replace('_', ' '), fontsize=10)
    axes[i].axis('off')
axes[-1].axis('off')
plt.suptitle('One sample image per lizard species', y=1.02)
plt.tight_layout(); plt.show()

### 2.4 Image sizes

Pre-trained CNNs expect a fixed input size. EfficientNetV2S was trained on 224×224 images, so that's the size we will resize to. But before deciding *how* to resize, we should know how big the originals are. If they were already 224×224, no work; if they were tiny (e.g. 64×64) we would be upscaling and creating fake detail; if they are huge we are downsampling and need to choose an appropriate interpolation method. We sample 30 images per class to keep this fast — measuring every single image is unnecessary just to get the order of magnitude.

In [ ]:
from PIL import Image as PILImage

widths, heights = [], []
for cname in CLASS_NAMES:
    cdir = os.path.join(TRAIN_DIR, cname)
    for fname in os.listdir(cdir)[:30]:
        with PILImage.open(os.path.join(cdir, fname)) as im:
            widths.append(im.size[0]); heights.append(im.size[1])

print(f'Width  range: {min(widths)} – {max(widths)}  (median {int(np.median(widths))})')
print(f'Height range: {min(heights)} – {max(heights)}  (median {int(np.median(heights))})')

**Observation:** Images are large (~1500–2000 px on each side). We resize all of them to 224×224, which is the resolution EfficientNetV2S was originally pretrained on. Downsampling from large images is generally safe — we lose fine detail, but the network's pretraining already optimised it for inputs of exactly this size.

---
## 3. Data Preparation

We need to turn folders of JPEGs into something the model can train on: a `tf.data.Dataset` that yields batches of `(image_tensor, integer_label)` pairs. The Keras helper `image_dataset_from_directory` does exactly this — it walks the directory tree, infers labels from subfolder names, decodes and resizes each JPEG, and batches them.

On top of that, we build a small **augmentation block** (random flips, rotations, zooms, etc.). With only ~1300 images, augmentation is what stops the model from memorising the training set. We place the augmentation layers *inside* the model so they run automatically during `model.fit()` and are bypassed at inference.

### 3.1 Build the train and validation datasets

`image_dataset_from_directory` is the modern Keras way to load an image folder. We call it twice with the same parameters but different `subset` values, so both datasets come from the same underlying split — making the validation set a proper held-out portion that the training set never sees. A few details worth flagging:

- `label_mode='int'` makes the labels integers, which pairs with `sparse_categorical_crossentropy` (no manual one-hot encoding needed).
- `class_names=CLASS_NAMES` locks the label order to the alphabetical order of the folders — without this argument, a different filesystem could return folders in a different order and silently swap labels.
- `validation_split=0.15` gives us an 85/15 train/val split. With ~1300 images we want as much training data as we can get, but we still need ~200 validation images for the val accuracy / F1 to be reliable.
- `shuffle=True` for training, `shuffle=False` for validation — keeping the validation order stable lets us cleanly concatenate `y_true` and the prediction array later.
- `cache()` keeps decoded images in RAM (only ~1300, easy fit). `prefetch(AUTOTUNE)` overlaps the next batch's loading with the current batch's training, removing I/O wait time. Both lines together typically halve the per-epoch time on small datasets.

In [ ]:
VALIDATION_SPLIT = 0.15

train_ds = image_dataset_from_directory(
    directory=TRAIN_DIR,
    labels='inferred',
    label_mode='int',
    class_names=CLASS_NAMES,
    batch_size=BATCH_SIZE,
    image_size=(IMG_SIZE, IMG_SIZE),
    validation_split=VALIDATION_SPLIT,
    subset='training',
    seed=SEED,
    shuffle=True,
)

val_ds = image_dataset_from_directory(
    directory=TRAIN_DIR,
    labels='inferred',
    label_mode='int',
    class_names=CLASS_NAMES,
    batch_size=BATCH_SIZE,
    image_size=(IMG_SIZE, IMG_SIZE),
    validation_split=VALIDATION_SPLIT,
    subset='validation',
    seed=SEED,
    shuffle=False,
)

print('Detected class order:', train_ds.class_names)

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().prefetch(AUTOTUNE)
val_ds   = val_ds.cache().prefetch(AUTOTUNE)

### 3.2 Define the augmentation block

We define augmentation as a `Sequential` model of stochastic layers. Each layer applies a random transformation to its input — but only when called with `training=True`, so at inference time the model sees the original image untouched. Plugging this block as the first stage of our main model means augmentation runs on the GPU as part of the same compute graph (faster than doing it in the data pipeline), and it is automatically inert at prediction time.

The specific augmentations are chosen to match how lizard photos vary in the wild:
- **Horizontal flip** — lizards have left/right symmetry, so a flipped photo is still a valid lizard photo. Free 2× data.
- **Rotation up to 10%** (≈ ±36°) — photographers don't always hold the camera level.
- **Zoom up to 15%** — simulates being closer or farther from the animal.
- **Translation up to 8%** — the lizard isn't always perfectly centred.
- **Contrast up to 15%** — sun and shade vary dramatically in nature shots.

We deliberately do *not* include vertical flips: an upside-down lizard is an unnatural image the test set never contains, so generating them would waste model capacity learning a useless distribution.

In [ ]:
data_augmentation = keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.10),
    layers.RandomZoom(0.15),
    layers.RandomTranslation(0.08, 0.08),
    layers.RandomContrast(0.15),
], name='data_augmentation')

### 3.3 Augmentation sanity check

Augmentation parameters are very easy to over-tune. If we set `RandomRotation(0.5)` (= ±180°) we would generate upside-down lizards; an aggressive `RandomContrast(0.9)` could flatten the image into something unrecognisable. By visualising 8 augmented samples we confirm three things at a glance:

1. **The lizards remain recognisable** — augmentations are not too aggressive.
2. **There is enough variation** between samples — augmentations are not too timid; consecutive epochs really do see different images.
3. **No bug in the pipeline** — for example, we are not accidentally producing pure black or pure white images.

This visualisation is *purely diagnostic*. It runs once, in this cell, and the images shown are not stored or fed back into training. If you removed this cell entirely the training would be identical.

In [ ]:
for imgs, lbls in train_ds.take(1):
    aug = data_augmentation(imgs, training=True)
    aug = tf.clip_by_value(aug, 0, 255).numpy().astype('uint8')
    fig, axes = plt.subplots(2, 4, figsize=(13, 6))
    for i, ax in enumerate(axes.flatten()):
        ax.imshow(aug[i])
        ax.set_title(CLASS_NAMES[int(lbls[i])].replace('_', ' '), fontsize=9)
        ax.axis('off')
    plt.suptitle('Augmented training samples (diagnostic only)', y=1.02)
    plt.tight_layout(); plt.show()
    break

---
## 4. Model — Transfer Learning with EfficientNetV2S

With only ~1300 training images, training a deep CNN from scratch is hopeless — the network would overfit immediately or never learn meaningful features. **Transfer learning** is the way around this: we start from a network that was already trained on ImageNet (1.2 million images, 1000 classes), and only retrain the *head* of that network for our 7 lizard species. The lower layers — which detect edges, textures, and shape primitives — are general enough to transfer to any natural-image task with no changes.

**Why EfficientNetV2S specifically?** It is a state-of-the-art, ImageNet-pretrained backbone with 513 layers and ~21 million parameters. It hits ~83% top-1 accuracy on ImageNet — meaning its feature maps capture genuinely strong visual structure. Compared with older architectures like ResNet50 or VGG, it gives noticeably higher accuracy for the same compute.

**Alternatives we considered:**
- **ResNet50V2** — older, proven backbone, slightly lower accuracy ceiling.
- **EfficientNetV2B0** — about 3× smaller, 2× faster, but the accuracy ceiling drops by 1–2 points.
- **MobileNetV3** — designed for mobile inference, very fast but the accuracy ceiling is too low for fine-grained classification like ours.

EfficientNetV2S strikes the best accuracy/speed balance for our problem.

### 4.1 Load the pre-trained backbone

We load EfficientNetV2S with the ImageNet weights. Two arguments are crucial:

- `include_top=False` removes the original 1000-class classifier (the layer that picks between "pug", "airliner", etc. on ImageNet). We will replace it with our own 7-class head.
- `weights='imagenet'` downloads the publicly available pre-trained weights. Without this we would be starting from random initialisation, which defeats the entire point of transfer learning.

We then set `base_model.trainable = False` to **freeze** all of the backbone's weights. In phase 1 of training, only the new head we add later will receive gradient updates.

In [ ]:
base_model = EfficientNetV2S(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,
    weights='imagenet',
)
base_model.trainable = False
print(f'EfficientNetV2S has {len(base_model.layers)} layers.')

### 4.2 Build the full model

We assemble the full forward pass: input image → augmentation → preprocessing → frozen backbone → classification head. Each component plays a specific role:

1. **`data_augmentation(inputs)`** — applies the random transformations defined above. Active only during training.
2. **`preprocess_input(x)`** — applies the exact normalisation EfficientNetV2S was pretrained with. Skipping or mismatching this step is a classic bug that quietly destroys accuracy.
3. **`base_model(x, training=False)`** — the frozen feature extractor. Note the explicit `training=False`: this keeps all `BatchNormalization` layers in inference mode (using the running ImageNet statistics) instead of recomputing batch stats on our small lizard batches. **Forgetting this flag is the single most common transfer-learning bug** — without it, BN statistics drift even when the layers are "frozen", and accuracy collapses.
4. **`GlobalAveragePooling2D`** — turns the 7×7×1280 feature map into a 1280-dimensional vector. It has zero parameters and is more robust to overfitting than a `Flatten` + huge `Dense` layer would be.
5. **Two `Dropout(0.3)` layers + a `Dense(128, relu)`** — a small bottleneck classifier head. Dropout randomly zeros 30% of activations during training, forcing the network to not rely too heavily on any single feature.
6. **`Dense(7, softmax)`** — the final classification layer, one neuron per species, softmax outputs that sum to 1.

We use the **Functional API** rather than `Sequential` precisely because we need that `training=False` argument on the base model — `Sequential` does not let us pass per-layer kwargs.

In [ ]:
inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3), name='image')
x = data_augmentation(inputs)
x = preprocess_input(x)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D(name='gap')(x)
x = layers.Dropout(0.3, name='drop_1')(x)
x = layers.Dense(128, activation='relu', name='dense_128')(x)
x = layers.Dropout(0.3, name='drop_2')(x)
outputs = layers.Dense(NUM_CLASSES, activation='softmax', name='predictions')(x)

model = keras.Model(inputs, outputs, name='lizard_classifier')
model.summary(line_length=110)

---
## 5. Training — Two Phases

Transfer learning works best when split into two phases:

1. **Phase 1 — Feature extraction.** The entire backbone is frozen; only the new head is trained. This is fast (only ~165k trainable parameters) and gives the head a sensible starting point. Without this phase, jumping straight to fine-tuning would feed *random* head outputs into the optimiser and could destabilise the carefully-pretrained backbone weights.
2. **Phase 2 — Fine-tuning.** We unfreeze the top 413 layers of the backbone (keeping the first 100 frozen) and continue training with a much smaller learning rate. The backbone's high-level features now slowly adapt to lizard-specific patterns.

Splitting the training this way gives us reliably better accuracy than either approach on its own.

### 5.1 Phase 1 — Feature extraction (frozen base)

We compile the model with `Adam(1e-3)` (a standard learning rate for training a fresh head) and `sparse_categorical_crossentropy` (the right loss for integer-encoded labels). We train for up to 12 epochs but rely on two callbacks to manage the run:

- **`EarlyStopping`** monitors validation accuracy. If it doesn't improve for 4 consecutive epochs, training stops early and `restore_best_weights=True` rewinds the model to the best epoch's weights. This both saves time and prevents late-stage overfitting.
- **`ModelCheckpoint`** saves the best-so-far weights to disk. If the kernel crashes mid-run we don't lose progress, and we can also reload this checkpoint later for diagnostics.

In [ ]:
model.compile(
    optimizer=optimizers.Adam(learning_rate=LR_P1),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)

ckpt_p1 = os.path.join(OUTPUT_DIR, 'best_phase1.keras')
callbacks_p1 = [
    EarlyStopping(monitor='val_accuracy', patience=4,
                  restore_best_weights=True, verbose=1),
    ModelCheckpoint(ckpt_p1, monitor='val_accuracy',
                    save_best_only=True, verbose=0),
]

history_p1 = model.fit(
    train_ds, validation_data=val_ds,
    epochs=EPOCHS_P1,
    callbacks=callbacks_p1,
    verbose=1,
)

### 5.2 Plot Phase 1 curves

Looking at training and validation curves tells us things a single accuracy number cannot:

- **Train accuracy ≫ validation accuracy** → overfitting. The augmentation or regularisation is too weak.
- **Train accuracy ≈ validation accuracy and both still rising** → train for more epochs.
- **Both flat and low** → the learning rate is too small, or the model has saturated.
- **Validation loss going up while accuracy still rising** → the model is becoming overconfident on wrong predictions; a sign of incipient overfitting.

We define a small helper `plotLosses` so we can reuse the same plotting code for phase 2.

In [ ]:
def plotLosses(history, title=''):
    """Loss + accuracy side by side."""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))
    ax1.plot(history.history['loss'],     label='training loss')
    ax1.plot(history.history['val_loss'], label='validation loss')
    ax1.set_title(f'Loss curves {title}')
    ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss'); ax1.legend()
    ax2.plot(history.history['accuracy'],     label='training accuracy')
    ax2.plot(history.history['val_accuracy'], label='validation accuracy')
    ax2.set_title(f'Accuracy curves {title}')
    ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy'); ax2.legend()
    plt.tight_layout(); plt.show()

plotLosses(history_p1, '— Phase 1 (frozen base)')

### 5.3 Unfreeze the top of the base for fine-tuning

Now we partially unfreeze the backbone. We set `base_model.trainable = True`, then re-freeze the first 100 layers — these are the lowest-level feature detectors (edges, simple textures, colour blobs). They generalise extremely well from ImageNet to any natural-image task, so updating them with our tiny dataset would only damage them.

Crucially, we also force every `BatchNormalization` layer in the entire backbone to stay frozen, even the ones in the unfrozen region. BN layers maintain running statistics (mean and variance) that were carefully tuned on millions of ImageNet images. If we let them update on our small batches of lizard images, those statistics become noisy and unstable, and accuracy collapses. This is one of the most well-documented but easy-to-miss pitfalls in transfer learning.

After this cell, only the upper ~413 layers of the backbone (excluding their BN layers) plus our new head are trainable — about 13 million parameters.

In [ ]:
base_model.trainable = True
for layer in base_model.layers[:FINE_TUNE_AT]:
    layer.trainable = False
for layer in base_model.layers:
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False

trainable = sum(np.prod(v.shape) for v in model.trainable_weights)
frozen    = sum(np.prod(v.shape) for v in model.non_trainable_weights)
print(f'Trainable params  : {int(trainable):,}')
print(f'Frozen params     : {int(frozen):,}')

### 5.4 Cosine learning-rate schedule

Fine-tuning is sensitive to the learning rate. A constant rate is the simple choice, but we get measurably better accuracy with a **cosine schedule with warmup**. The schedule has two phases:

1. **Warmup (epochs 0–1):** the LR ramps linearly from `0` up to `1e-4`. This prevents a sudden large gradient step on the freshly-unfrozen layers, which could blow up the carefully-pretrained weights.
2. **Cosine decay (epochs 2–24):** the LR follows the right half of a cosine curve from `1e-4` down to `1e-6`. The high LR at the start lets the model leave a poor local minimum; the low LR at the end lets it settle precisely into a better one.

Empirically, cosine decay gives 0.5–1.5 accuracy points over a constant LR on small image classification problems. The Kaggle brief explicitly mentions "learning rate schedules" as a bonus enhancement, so this also satisfies that. We plot the schedule to make the shape obvious.

In [ ]:
WARMUP = 2
def cosine_lr(epoch, _lr):
    if epoch < WARMUP:
        return float(LR_P2_MAX * (epoch + 1) / WARMUP)
    progress = (epoch - WARMUP) / max(1, EPOCHS_P2 - WARMUP)
    return float(LR_P2_MIN + 0.5 * (LR_P2_MAX - LR_P2_MIN) *
                 (1 + math.cos(math.pi * progress)))

lrs = [cosine_lr(e, None) for e in range(EPOCHS_P2)]
plt.figure(figsize=(7, 2.8))
plt.plot(lrs, marker='o', ms=4)
plt.title('Cosine LR schedule (Phase 2)')
plt.xlabel('Epoch'); plt.ylabel('LR'); plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()

### 5.5 Phase 2 — Fine-tune

We **must** call `model.compile()` again here, even though we already compiled in phase 1. Whenever you change any `trainable` flag in Keras, you have to recompile so the new graph (with its new set of trainable variables) is picked up. Otherwise the optimiser still uses the old set and the unfrozen layers never actually receive updates.

We then train for up to 25 epochs with three callbacks: `EarlyStopping` (with `patience=8` because fine-tuning is noisier than feature extraction), `ModelCheckpoint`, and the `LearningRateScheduler` we just defined. The cosine schedule will quietly adjust the LR every epoch.

In [ ]:
model.compile(
    optimizer=optimizers.Adam(learning_rate=LR_P2_MAX),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)

ckpt_p2 = os.path.join(OUTPUT_DIR, 'best_phase2.keras')
callbacks_p2 = [
    EarlyStopping(monitor='val_accuracy', patience=8,
                  restore_best_weights=True, verbose=1),
    ModelCheckpoint(ckpt_p2, monitor='val_accuracy',
                    save_best_only=True, verbose=0),
    LearningRateScheduler(cosine_lr, verbose=0),
]

history_p2 = model.fit(
    train_ds, validation_data=val_ds,
    epochs=EPOCHS_P2,
    callbacks=callbacks_p2,
    verbose=1,
)

### 5.6 Phase 2 + combined training history

We plot phase 2 alone, then the combined history of both phases on a single chart with a vertical dashed line marking the boundary. The combined plot is the one to study: typically you see a clear jump in accuracy right after the boundary, which is the textbook sign that fine-tuning is doing what it's supposed to. If the curves stay flat across the boundary, fine-tuning didn't help — usually because the LR is too small (or BN was not properly frozen).

In [ ]:
plotLosses(history_p2, '— Phase 2 (fine-tuning)')

h1, h2 = history_p1.history, history_p2.history
loss_all  = h1['loss']     + h2['loss']
vloss_all = h1['val_loss'] + h2['val_loss']
acc_all   = h1['accuracy']     + h2['accuracy']
vacc_all  = h1['val_accuracy'] + h2['val_accuracy']
boundary = len(h1['loss'])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))
ax1.plot(loss_all, label='train'); ax1.plot(vloss_all, label='val')
ax1.axvline(boundary - 0.5, ls='--', c='gray', alpha=0.7)
ax1.set_title('Loss — full training'); ax1.legend()
ax2.plot(acc_all, label='train'); ax2.plot(vacc_all, label='val')
ax2.axvline(boundary - 0.5, ls='--', c='gray', alpha=0.7)
ax2.set_title('Accuracy — full training'); ax2.legend()
plt.tight_layout(); plt.show()

print(f'\nBest validation accuracy seen: {max(vacc_all):.4f}')

### 5.7 Optimal epoch analysis

We trained for up to `EPOCHS_P1 + EPOCHS_P2 = 37` epochs in total, but the *best* model is rarely the last one — usually it's somewhere in the middle. To make the optimal-epoch decision visible and defensible, we plot two diagnostic charts side by side:

1. **Validation accuracy across all epochs**, with a red dashed line marking the peak. The peak is the model `EarlyStopping` ultimately restores. If the peak is at the very last epoch, training was probably cut short and we should consider raising `patience`. If it's in the middle followed by clear decline, that is exactly the overfitting signal `EarlyStopping` is designed to catch.
2. **Train accuracy − Validation accuracy gap**, the standard overfitting indicator. A small positive gap is healthy (the model fits training data slightly better than unseen data). A widening gap means the model is starting to memorise specifics of the training set that don't generalise.

Both plots also show the phase 1 → phase 2 boundary as a faint grey dashed line so we can see whether the gain came from the head training or the fine-tuning.

In [ ]:
best_epoch  = int(np.argmax(vacc_all))
best_val    = float(vacc_all[best_epoch])
total_epochs = len(vacc_all)
gap = np.array(acc_all) - np.array(vacc_all)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4.8))

# Plot 1: validation accuracy with best-epoch marker
ax1.plot(range(total_epochs), vacc_all, 'o-', color='steelblue',
         label='validation accuracy')
ax1.axvline(best_epoch, ls='--', c='red', alpha=0.7,
            label=f'optimal epoch = {best_epoch + 1}')
ax1.axhline(best_val,  ls=':',  c='red', alpha=0.5,
            label=f'best val acc = {best_val:.4f}')
ax1.axvline(boundary - 0.5, ls='--', c='gray', alpha=0.4,
            label='phase 1 → 2 boundary')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Validation accuracy')
ax1.set_title('Optimal epoch — peak validation accuracy')
ax1.legend(loc='lower right'); ax1.grid(alpha=0.3)

# Plot 2: train/val accuracy gap (overfitting indicator)
ax2.plot(range(total_epochs), gap, 'o-', color='darkorange',
         label='train acc − val acc')
ax2.axhline(0, ls='-', c='gray', alpha=0.4)
ax2.axvline(best_epoch, ls='--', c='red', alpha=0.7,
            label=f'optimal epoch = {best_epoch + 1}')
ax2.axvline(boundary - 0.5, ls='--', c='gray', alpha=0.4,
            label='phase 1 → 2 boundary')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Train acc − Val acc')
ax2.set_title('Overfitting gap — bigger = more overfitting')
ax2.legend(loc='upper left'); ax2.grid(alpha=0.3)

plt.tight_layout(); plt.show()

print(f'\nOptimal epoch        : {best_epoch + 1} of {total_epochs}')
print(f'Best val accuracy    : {best_val:.4f}')
print(f'Train/val gap there  : {gap[best_epoch]:+.4f}')
print('\nEarlyStopping already restored the model to this epoch\'s weights.')

---
## 6. Model Evaluation

Now that training is finished and `EarlyStopping` has rewound the model to its best epoch, we evaluate it on the held-out validation set with four complementary tools:

1. **Validation accuracy** — the headline number, easy to interpret.
2. **Confusion matrix + F1-macro** — the matrix is required by the brief and shows *which* species get confused; F1-macro is the actual Kaggle scoring metric.
3. **Per-class precision / recall / F1 report** — detailed breakdown so we can tell if a poor F1-macro is one bad class dragging down the average.
4. **Most-confidently-wrong predictions** — qualitative inspection of the hardest cases.

### 6.1 Validation accuracy

We re-evaluate on the validation set with `model.evaluate()`. The returned accuracy should match the peak we identified in section 5.7 (since `EarlyStopping` restored those weights). If they disagree by more than a tiny margin, something went wrong — for example, the model checkpoint was overwritten or the validation dataset was shuffled.

In [ ]:
val_loss, val_acc = model.evaluate(val_ds, verbose=0)
print(f'Validation loss     : {val_loss:.4f}')
print(f'Validation accuracy : {val_acc:.4f}')

### 6.2 Confusion matrix and F1-macro

**Why F1-macro and not just accuracy?** Accuracy treats every test image as equally important, which means it can hide a systematic failure on a single class. F1-macro instead averages the per-class F1 scores, treating each *class* equally — so getting one class consistently wrong drops F1-macro by ~14 points (1/7) even if overall accuracy stays high. Since Kaggle scores us on F1-macro, this is the number we need to optimise.

**The confusion matrix** shows the count of (true class, predicted class) pairs. The diagonal is correct predictions; off-diagonal entries are mistakes. The matrix is the most actionable evaluation tool we have — if we wanted to spend more time improving the model, the very first place to look is *which* off-diagonal cell is highest, and *why* the model confuses those two species. Often it points at a labelling issue, a dataset bias, or a missing augmentation.

In [ ]:
y_true = np.concatenate([y for _, y in val_ds], axis=0)
y_prob = model.predict(val_ds, verbose=0)
y_pred = np.argmax(y_prob, axis=1)

f1_macro = f1_score(y_true, y_pred, average='macro')
print(f'Validation F1 (macro): {f1_macro:.4f}   ← Kaggle scoring metric')

cm = confusion_matrix(y_true, y_pred)
short = [n.replace('_', ' ') for n in CLASS_NAMES]
disp = ConfusionMatrixDisplay(cm, display_labels=short)
fig, ax = plt.subplots(figsize=(8, 6.5))
disp.plot(cmap=plt.cm.Blues, ax=ax, xticks_rotation=35)
ax.set_title('Confusion matrix (validation set)')
plt.tight_layout(); plt.show()

### 6.3 Per-class precision / recall / F1

`classification_report` prints precision, recall, and F1 for each class plus macro and weighted averages. The two columns answer different questions:

- **Precision** — of all images the model labelled "X", what fraction really were X? Low precision means the model over-predicts that class.
- **Recall** — of all images that really are X, what fraction did the model catch? Low recall means the model misses that class.

Each failure mode requires a different fix. Low recall on a class often points at insufficient training examples or augmentation that's too aggressive for that class's appearance. Low precision usually points at confusable look-alikes.

In [ ]:
print(classification_report(y_true, y_pred, target_names=short, digits=3))

### 6.4 Most-confidently-wrong predictions

We look at the validation images where the model was most *confident* and still *wrong* — the highest-confidence mistakes. These are by far the most informative errors to inspect:

- If a label is *clearly correct in the photo* and the model confidently disagrees, the model has a real blind spot — perhaps an unusual angle, lighting, or background that wasn't well-represented in training.
- If the *photo is genuinely ambiguous* or *the label looks wrong*, that's a dataset issue, not a model issue.

Either way, this is the cheapest qualitative diagnostic available. We show the top 8.

In [ ]:
max_conf = y_prob.max(axis=1)
wrong = np.where(y_pred != y_true)[0]
wrong_sorted = wrong[np.argsort(-max_conf[wrong])][:8]

val_imgs = np.concatenate([imgs.numpy() for imgs, _ in val_ds], axis=0)

if len(wrong_sorted) > 0:
    fig, axes = plt.subplots(2, 4, figsize=(14, 7))
    for ax, idx in zip(axes.flatten(), wrong_sorted):
        ax.imshow(val_imgs[idx].astype('uint8'))
        ax.set_title(
            f'true: {short[y_true[idx]]}\npred: {short[y_pred[idx]]} '
            f'({max_conf[idx]:.2f})', fontsize=9)
        ax.axis('off')
    plt.suptitle('Most-confidently-wrong validation predictions', y=1.02)
    plt.tight_layout(); plt.show()
else:
    print('No wrong predictions found 🎉')

---
## 7. Test Predictions with TTA

Now that we trust the model, we run it on the test set to generate the Kaggle submission. Instead of doing a single forward pass per image, we use **Test-Time Augmentation (TTA)**: each test image is fed through the model twice — once as-is, and once horizontally flipped. We average the two softmax outputs and take the argmax of the average.

This works because lizards are left/right symmetric: the original and flipped photo are equally valid views of the same animal. If the model is uncertain on a particular pose, the flipped view gives an independent vote. Averaging two predictions reduces variance with no additional training cost. Empirically, this gains 1–2 F1 points almost every time.

Heavier TTA (rotations, crops, more flips) gives diminishing returns and risks introducing augmentations that change what the model thinks the image is. The 2-pass version is the safe sweet spot.

### 7.1 Build test pipeline and run TTA

We build a `tf.data.Dataset` of decoded test images, cache it in RAM (so we only decode each JPEG once), and create two views: the original and a horizontally-flipped copy. We call `model.predict()` on each, then average the softmax probabilities element-wise. The argmax of the average is our final predicted class. The cache + flip-on-the-fly approach is fast because flipping a tensor is a few orders of magnitude cheaper than re-decoding a JPEG.

In [ ]:
test_ids = test_df['id'].tolist()
test_paths = [os.path.join(TEST_DIR, f'{i}.jpg') for i in test_ids]
missing = [p for p in test_paths if not os.path.exists(p)]
print(f'Test images : {len(test_paths)}    missing: {len(missing)}')

def decode_test(path):
    img = tf.io.read_file(path)
    img = tf.io.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    return tf.cast(img, tf.float32)

raw_test = (tf.data.Dataset.from_tensor_slices(test_paths)
            .map(decode_test, num_parallel_calls=AUTOTUNE)
            .cache())

test_ds_orig = raw_test.batch(BATCH_SIZE).prefetch(AUTOTUNE)
test_ds_flip = (raw_test
                .map(tf.image.flip_left_right, num_parallel_calls=AUTOTUNE)
                .batch(BATCH_SIZE).prefetch(AUTOTUNE))

probs_orig = model.predict(test_ds_orig, verbose=1)
probs_flip = model.predict(test_ds_flip, verbose=1)
probs_avg  = (probs_orig + probs_flip) / 2.0
test_pred  = np.argmax(probs_avg, axis=1)
print(f'\nGenerated {len(test_pred)} test predictions.')

### 7.2 Sanity-check the predicted distribution

The Kaggle test set was sampled the same way as the training set, so we expect roughly the same class balance — about 326 / 7 ≈ 47 predictions per class. If the model predicted, say, 250 of one class, that would indicate something went badly wrong upstream (a label mix-up, broken preprocessing, etc.). This is a cheap last check before we write the submission file.

In [ ]:
submission = pd.DataFrame({'id': test_ids, 'label': test_pred.astype(int)})

pred_counts = submission['label'].value_counts().sort_index()
print('Predicted class distribution on the test set:')
for i in range(NUM_CLASSES):
    print(f'  {i} ({CLASS_NAMES[i]:<28}): {pred_counts.get(i, 0)}')

fig, ax = plt.subplots(figsize=(11, 3.5))
ax.bar(range(NUM_CLASSES),
       [pred_counts.get(i, 0) for i in range(NUM_CLASSES)],
       color='seagreen')
ax.set_xticks(range(NUM_CLASSES))
ax.set_xticklabels(short, rotation=20, ha='right')
ax.set_title('Predicted class counts on the test set')
plt.tight_layout(); plt.show()

### 7.3 Save the submission file

We write the submission as a CSV with two columns — `id` and `label` — exactly matching the `sample_submission.csv` format. We save two copies: one in `project_outputs/` alongside our checkpoints, and one in the project root for easy uploading. Both contain identical content.

In [ ]:
out_path = os.path.join(OUTPUT_DIR, 'submission.csv')
submission.to_csv(out_path, index=False)
submission.to_csv('submission.csv', index=False)
print(f'Saved : {out_path}')
print(f'Saved : submission.csv (project root)')
submission.head(10)

---
## 8. GenAI Section

*(Required by the brief.)*

### How we used GenAI

We treated GenAI as a **pair-programmer and reviewer**, not as an oracle. Concretely, we used it in four ways:

1. **Skeleton scaffolding.** We asked an LLM to generate the initial notebook skeleton — imports, EDA template, training loop boilerplate — so we could spend our time on *decisions* (which model, which hyper-parameters, which augmentations) rather than on retyping standard code.

2. **Debugging silent bugs.** An early version of our model scored only 0.72 on Kaggle — clearly something was wrong. The LLM correctly flagged that newer EfficientNet variants in `tf.keras.applications` already include internal `Rescaling` and `Normalization` layers, so calling `preprocess_input` again on top would double-normalise the inputs. We verified this by reading the official TensorFlow source and the regression test file. That single fix accounted for most of the accuracy gap.

3. **Method comparisons.** Whenever we faced a design choice — cosine LR vs constant LR, EfficientNetV2S vs V2B0, TTA vs CV ensembles — we asked the LLM to lay out the trade-offs. We then made the decision ourselves based on our compute budget and accuracy target. The LLM was good at *enumerating* options; it was less reliable at predicting *which* option would win on our specific problem.

4. **Plain-English explanations.** When we encountered an unfamiliar concept (F1-macro, label smoothing, mixed-precision training), we asked for an intuitive explanation and then verified it against external sources before relying on it.

### What we did *not* let GenAI do

- We did not let it pick the final hyper-parameters blindly — we tested them ourselves and looked at the curves.
- We did not let it write our defense; every cell in this notebook we can explain ourselves.
- We did not paste in answers without reading them first. Every chunk of generated text we edited and verified before keeping.

### Reflection

GenAI was most useful in three places: catching silent bugs that humans miss (the double-normalisation), summarising idioms across documentation, and suggesting small enhancements (cosine schedule, TTA) that turned a 0.72 baseline into a much stronger result. It was weakest at open-ended judgement calls — when we asked things like *"is this good enough?"* it tended to say yes regardless. We learned to ask **falsifiable questions** instead: "what specific bug could explain accuracy of ~70% on a balanced 7-class problem?" produces useful answers; "is my model good?" does not.

---
## 9. Summary

| Component | Choice |
|---|---|
| Backbone | EfficientNetV2S (ImageNet) |
| Input size | 224 × 224 |
| Loader | `image_dataset_from_directory` + cache + prefetch |
| Augmentation | In-model: flip / rotation / zoom / translation / contrast |
| Phases | (1) frozen base → (2) `fine_tune_at = 100` |
| LR (phase 2) | Cosine with 2-epoch warmup |
| TTA | 2 passes (original + horizontal flip) |

**Outputs (in `project_outputs/`):**
- `best_phase1.keras`, `best_phase2.keras` — checkpoints
- `submission.csv` — Kaggle submission

**To push the score higher (after submitting once):**
1. **5-fold CV ensemble** — re-run with 5 different seeds and average the test-set softmax outputs. Reliably gives +1–2 F1 points but takes 5× the training time.
2. **Bigger input (260 / 300 px)** — a larger receptive field captures more detail. Typically +1 point at modest extra cost.
3. **Label smoothing of 0.05** in the loss function — turns each one-hot target into a softer distribution. Typically +0.3 to +0.7 points on small classification problems.
4. **More TTA passes** with mild rotations and crops — diminishing returns kick in after about 5 passes, but the first few are essentially free.